# Datamine PROMOD Process Example

This notebook demonstrates how to configure and run the **`promod`** process wrapper in `dmstudio`.

## Process Description

## PROMOD

Optimize a block model so that the minimum number of subcells is used, without losing accuracy.

The optimization is achieved by combining adjacent subcells within the same cell. The user controls this operation by defining fields that must be the same (or within a specified tolerance) before two subcells may be combined.

**PROMOD** also checks the input block model for "errors". These "errors" include a subcell extending outside the co-ordinate limits of its cell and two subcells overlapping each other.

The process requires a block model file as input. It produces the optimized block model and optionally a remnants file.

### Methodology

The overall logic flow for the PROMOD process is as follows:

1\. Check subcell for extension outside its cell co-ordinate limits (see below).
2\. Check whether any two subcells in a cell overlap each other (see below).
3\. Optimize the subcells within a cell (see below).

### Extension of Subcell Outside its Cell

In a Datamine block model it is invalid to have a subcell extending outside its cell. However, this error may occur where a model has been altered "manually" (e.g. using [EXTRA](<extra.md>)). **PROMOD** handles such errors in the manner detailed below.

Every subcell has its X co-ordinate minimum and maximum calculated. If either of these limits is found to exceed that of the cell, the subcell is truncated to the cell edge (see Figure 1). The portion(s) of the original subcell outside the cell will be written to the optional &**REMNANTS** file.

This process is then repeated for the Y and the Z co-ordinates. Thus if the original subcell exceeded the cell in all six directions, six records will be output to the &REMNANTS file.

A single warning message is given if one or more subcells extend outside their cells. Details of every occurrence of a subcell extending outside its cell may be listed by setting @PRINT=3.

### Overlapping subcells

In a Datamine block model it is invalid to have any two subcells overlapping each other. However, this error may occur where a model has been altered "manually" (e.g. using [EXTRA](<extra.md>)). **PROMOD** handles such errors in the manner described below, dependent on the @**OVERLAP** parameter.

Details of every overlap of subcells may be obtained by setting @**PRINT** =1 and @**OVERLAP** = 1 or 2

## #### @OVERLAP=0

If **PROMOD** encounters any two subcells t in any one or more of X, Y and Z, it will stop processing. Note that the first overlap will terminate **PROMOD** ; there may be others in the &IN model.

## #### @OVERLAP=1

Any overlap of subcells encountered in the &**IN** model will be ignored. The subcells will be unaltered and will be processed by the rest of **PROMOD** and written to the &**OUT** model. Thus the output model will still contain these errors and will, for instance, give misleading volumes, tones and grades.

## #### @OVERLAP=2

Any overlapping subcells will be resolved by splitting into more subcells as appropriate (see Figure 2). This is the same logic used by the [ADDMOD](<addmod.md>) process when combining two overlapping subcells from two separate models. Where the two overlapping subcells in **PROMOD** have a different value for the same field, the one which occurs second in the &IN model file takes precedence.

### Optimization

Within any cell, **PROMOD** may "optimize" the use of subcells so that the number is minimized without loss of geometrical or numerical accuracy. The degree of optimization that **PROMOD** performs is defined by the @**OPTIMISE** parameter. To obtain a list of subcell combinations performed set @**PRINT** =2.

## #### @OPTIMISE=2

Subcells will be combined with each other, to form one larger cell, if it is possible geometrically (e.g. they must at least touch each other) and if the values of the key field(s) in each of are consistent. The key fields are specified by the ten parameters * **KEY1** to * **KEY10**. For instance, consider four equally sized subcells making up one cell (see Figure 3).

Specifying * **KEY1**(**ROCK**) will allow subcell A to be combined with subcell B, but not with C or D since they have a different value for **ROCK** (see Figure 3(a)). The values of the fields in the output subcells are volume or tonnage weighted averages for numerical fields. For alphanumerical fields they are the dominant value; that is the value with the greatest volume or tonnage. If **PROMOD** is re-run on the same input model, but with two key fields (**ROCK** and **FE**), no subcells are combined because none have the same value for FE (see Figure 3(b)). However, if the @**TOL** parameter is set to 10 then subcells A and B are combined, but not C and D (see Figure 3(c)). This is because subcells are combined if their numerical key field values are within @TOL of each other. @**TOL** is defined as a percentage of the total range of each key field. For instance (as in Figure 3(c)), an @**TOL** of 10 for a model where the total range of FE values is 50 (max=70, min= 20) means the tolerance in FE is 5. Subcells will be combined if:

ROCK (subcell A) = ROCK (subcell B) AND

FE (subcell A) = FE (subcell B)+/-5

Note that the tolerance for different numerical fields will vary if their total ranges differ. For example, if P was also defined as a key field the criteria for subcell combination might become:

ROCK (subcell A) = ROCK (subcell B) AND

FE (subcell A) = FE (subcell B)+/-5 AND

P (subcell A) = P (subcell B)+/-0.047

The three parameters called @**XINCMIN** , @**YINCMIN** and @**ZINCMIN** also may have an effect on whether two subcells are combined. If either of the subcells has an X dimension (**XINC**) less than @**XINCMIN** , or **YINC** <@**YINCMIN** , or **ZINC** <@**ZINCMIN** then the key field value(s) will not be examined. In other words, the two subcells will be combined if geometrically possible, regardless of their values of key field(s) (see Figure 4). In effect, this allows the user to remove what are considered to be insignificantly small subcells. However, rather than just deleting them, which would produce gaps in the model, they are combined and volume/tonnage weighted averages calculated. Thus, the total volume of the model and the overall average grades etc. are unchanged.

## #### @OPTIMISE=1

Subcells will be combined if they result in a single cell in the cell. For the majority of cases, this means they will be combined only if they form a complete cell.

However, at the edges of the model, cells may not be totally filled with subcells. In this case, with @**OPTIMISE** =1, subcells will be combined if they create one subcell, even though it will not occupy all of the parent cell.

The logic used in the combination algorithm is the same as that used for @**OPTIMISE** =2.

## #### @OPTIMISE=0

No combination of subcells is performed at all. All subcells present in the input model, plus any created by overlap resolution, will be written to the output model.

### File Handling

#### &IN - Input Model

The input model must contain the standard 13 model fields and be sorted by IJK. If it has a field called DENSITY, it will be used to tonnage weight the calculated average and dominant values.

#### &OUT - Output Model

This is a standard model file with the same field names and defaults as the &IN input model. It will be sorted by IJK.

#### &REMNANTS - Remnants file

The output &**REMNANTS** file is a model file with the same field names and defaults as the &IN input model. It contains any portions of subcells lying outside their cell (see above).

**PROMOD** calculates the correct dimensions (i.e. **XINC** , **YINC** and **ZINC** field values) for these subcell remnants, but their IJK value remains unaltered. This follows their source parent cell to be identified. To assist in determining whether any remnants produced are "significant", their total volume can be simply calculated using **GENTRA** (with **MUL VOL XINC YINC;** **MUL VOL VOL ZINC** commands) and **STATS** (with * **F1(VOL)**). Note that the &**REMNANTS** file is not a valid model since all the IJK values are incorrect. Thus standard model processes, such as **MODLED** and **MODRES** , may give misleading results.

### Detailed Features

#### Density

If the &**IN** input model contains a field called **DENSITY** then it is used in calculations of average and dominant values when combining subcells. In other words, tonnage weighting is used. If a field name is specified as the * **DENSITY** field then it is used as the density.

If no * **DENSITY** field is specified and no field called **DENSITY** exists, the calculated average and dominant values are volume weighted.

Values calculated when combining subcells

It is often the case that several subcells are combined to form one subcell. For example, if no key fields were specified for the data shown in Figure 3, all four subcells would be combined into one. In such a case, the final calculated averages and dominant values are derived from all four contributing subcells at once.

In other words, the final single subcell's values are derived from all contributing subcells and not from any intermediary subcells (e.f. subcell A and B combined) resulting from previous combinations. This ensures that the final dominant value for an alphanumerical field is always correct.

### Displayed Output

The amount of information displayed by **PROMOD** during its processing is controlled by the @PRINT parameter.

## #### @PRINT=1

Details of every occurrence of overlapping subcells will be shown as here:

>>> Overlap in parent cell IJK 110

X,Y,Z limits:  |  105.000 |  110.000 |  105.000 |  110.000 |  100.000 |  110.000
---|---|---|---|---|---|---
X,Y,Z limits: |  100.000  |  107.000  |  105.000  |  110.000  |  100.000  |  110.000

## #### @PRINT=2

Details of every combination of subcells will be given as here:

>>> Combine cells in IJK=         110

* **Output** (|  105.000 |  105.000  |  105.000  |  10.000  |  10.000  |  10.000):
---|---|---|---|---|---|---
Input 1:  |  102.500  |  102.500  |  105.000  |  5.000  |  5.000  |  5.000
Input 2:  |  107.500  |  102.500  |  105.000  |  5.000  |  5.000  |  10.000
Input 3:  |  102.500  |  107.500  |  105.000  |  5.000  |  5.000  |  10.000
Input 4:  |  107.500  |  107.500  |  105.000  |  5.000  |  5.000  |  10.000

## #### @PRINT=3

Details of every occurrence of a subcell extending outside its parent cell will be shown as here:

>>>Subcell extends outside cell IJK 110 by 2.000

Cell limits: 100.000 110.000 Subcell limits: 105.000 112.000
Subcell XC,XINC,YC,YINC,ZC,ZINC: 108.500 7.000 107.500 5.000 105.000 10.000

## #### @PRINT=4

Details listed will be as for @**PRINT** =1 plus @**PRINT** = 2

## #### @PRINT=5

Details listed will be as for @**PRINT** =1 plus @**PRINT** = 2 plus @**PRINT** = 3

Combining Different Geological Codes

It is easy to get **PROMOD** to combine codes of the same value. For example, combination of **GEOZONE** values of 50 with 50, 51 with 51, 60 with 60 etc. is achieved by defining * **KEY1**(GEOZONE).

However, if it is desired to combine **GEOZONE** values of 50 with anything from 50-59, 60 with 60-69 etc a new category field must be set up prior to **PROMOD**. This is simple achieved in **GENTRA** with:

## SETC GEOCAT |  0

---|---

## GEC GEOZONE |  50

## LEC GEOZONE |  59

## SETC GEOCAT |  50

## GEC GEOZONE |  60

## LEC GEOZONE |  69

## SETC GEOCAT |  60

...and so on

**PROMOD** would then be run with * **KEY1**(GEOCAT). A similar approach would be used for alphanumerical codes (e.g. to combine SH with WST).

### Input Files:

* **in** (Block Model):
  Input model. Must contain at least the fields **XC, YC, ZC, XINC, YINC, ZINC, XMORIG,
  YMORIG, ZMORIG, NX, NY, NZ** , and **IJK**. May also contain value fields. It must be
  sorted by **IJK**.
  Required=Yes

### Output Files:

* **out** (Block Model):
  Output model. Will contain all the fields held in the IN file. It will be sorted by

## IJK.

  Required=Yes

* **remnants** (Block Model):
  Optional output model file holding remnants of any subcell outside its parent cell.
  Required=No

### Fields:

* **keys** (Undefined : Undefined):
  Field from the IN file which must be the same for two or more subcells to be combined if **OPTIMISE** =1 or 2.**Note** : 20 keyfields are supported if **PROMOD** is run interactively. 50 can be provided if run from a macro.
  Default=Undefined
  Required=No

* **density** (Undefined : IN):
  Field to weight values in OUT when combining subcells if **OPTIMISE** =1 or 2. If a
  field called **DENSITY** exists in the IN file it will be used.
  Default=Undefined
  Required=No

### Parameters:

* **density**:
  Density value used if **DENSITY** field value in a sub-cell is absent. Default (1.0).
  Range=Undefined
  Values=Undefined
  Default=1.0
  Required=No

* **xincmin**:
  Defines minimum subcell dimension in X. Any subcell with a dimension less than this
  will be combined (if **OPTIMISE** =1 or 2, and if possible) with an adjacent subcell
  regardless of key field{s} values. Default is parent cell X size / 1000.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **yincmin**:
  Defines minimum subcell dimension in Y. Any subcell with a dimension less than this
  will be combined (if **OPTIMISE** =1 or 2, and if possible) with an adjacent subcell
  regardless of key field{s} values. Default is parent cell Y size / 1000.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **zincmin**:
  Defines minimum subcell dimension in Z. Any subcell with a dimension less than this
  will be combined (if **OPTIMISE** =1 or 2, and if possible) with an adjacent subcell
  regardless of key field{s} values. Default is parent cell Z size / 1000.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **overlap**:
  Overlap checking and resolution. Default (0). Options: 0: \- Existence of an overlap in
  the IN model will be reported and **PROMOD** will terminate.; 1: \- Existence of an
  overlap in the model will be reported. No attempt will be made to resolve the overlaps.
  They will be copied into the **OUT** file.; 2: \- Overlaps will be resolved according to
  **ADDMOD** logic. 2nd subcell will have priority
  Range=0,2
  Values=0,1,2
  Default=0
  Required=No

* **optimise**:
  Optimise combination of subcells to minimise number. Default (2). Options: 0: \- No
  combination of subcells.; 1: \- Combination of subcells only if they form a complete
  parent cell.; 2: \- Combination of subcells to form minimum number of subcells.
  Range=0,2
  Values=0,1,2
  Default=2
  Required=No

* **tol**:
  Tolerance on numerical key field{s} comparison. Subcells with key numeric fields within
  **TOL** of each other may be combined if **OPTIMISE** =1 or 2. TOL is specified as a
  percentage of the range of values for each field. Default (0.001).
  Range=Undefined
  Values=Undefined
  Default=0.001
  Required=No

* **accuracy**:
  Accuracy indicates the size below which a cell is deemed to be invisible. Default
  (0.001). Note that these cells are ignored by the **PROMOD** process.
  Range=Undefined
  Values=Undefined
  Default=0.001
  Required=No

* **print**:
  Print flag. Default (0). Options: 0: \- minimum output.; 1: \- details of every overlap
  in the model.; 2: \- details of each combination of subcells.; 3: \- details of all
  subcells extending outside the parent cell limits.; 4: \- as 1 plus 2.; 5: \- as 1 plus
  2 plus 3.
  Range=0,5
  Values=0,1,2,3,4,5
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('promod')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute promod
print("Running promod...")
dm_cmd.promod(
    in_i='t_assays',  # required
    out_o='t_promod_out',  # required
    # remnants_o=['t_promod_out'],  # optional
    # keys_f=['BHID'],  # optional
    # density_f='optional',  # optional
    # density_p=1.0,  # optional
    # xincmin_p='optional',  # optional
    # yincmin_p='optional',  # optional
    # zincmin_p='optional',  # optional
    # overlap_p=0,  # optional
    # optimise_p=2,  # optional
    # tol_p=0.001,  # optional
    # accuracy_p=0.001,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("promod execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_promod_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")